### Interpolación lineal

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Corrected file paths (ensure they match your actual file locations)
file_path = Path('C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Ratios/en valores_ratios-trimestral_120 m.xlsx')
output_file_path = Path('C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Ratios/ratios_mensual_lineal.xlsx')

# Verify that the input file exists
if not file_path.exists():
    raise FileNotFoundError(f"The file was not found at the specified path: {file_path}")

# Load the Excel file
xls = pd.ExcelFile(file_path)

print("Sheets in the Excel file:", xls.sheet_names)

def interpolate_sheet(sheet_df):
    periods = sheet_df.columns[1:]  # Ignore the first column (company names)
    companies = sheet_df.iloc[:, 0]  # Company names

    N = len(periods)

    # Generate labels starting with M3 (end of quarter)
    monthly_periods = []
    for t in range(N - 1):
        monthly_periods.append(f"{periods[t]}_M3")       # Data point at end of current quarter
        monthly_periods.append(f"{periods[t+1]}_M1")     # First month of next quarter
        monthly_periods.append(f"{periods[t+1]}_M2")     # Second month of next quarter
    # Append the last quarter's M3
    monthly_periods.append(f"{periods[N - 1]}_M3")       # Data point at end of last quarter

    total_months = len(monthly_periods)

    # Initialize DataFrame with companies as index and monthly_periods as columns
    interpolated_df = pd.DataFrame(index=companies, columns=monthly_periods)

    # Indices of data points (every third month)
    data_times = np.arange(0, total_months, 3)

    for i, company in enumerate(companies):
        # Replace non-numeric entries with np.nan and convert to numeric
        values = sheet_df.iloc[i, 1:].replace({'-': np.nan, '—': np.nan}).values
        values = pd.to_numeric(values, errors='coerce')

        # Initialize the interpolated values with NaNs
        interpolated_values = np.full(total_months, np.nan)

        # Get indices where data is valid
        valid_indices = np.where(~np.isnan(values))[0]

        # Loop over segments of consecutive valid data
        for grp in np.split(valid_indices, np.where(np.diff(valid_indices) != 1)[0]+1):
            if len(grp) > 1:
                # There are consecutive valid points to interpolate between
                start_idx = grp[0]
                end_idx = grp[-1]
                x = data_times[start_idx:end_idx+1]
                y = values[start_idx:end_idx+1]

                # Time indices for interpolation within this segment
                interp_times = np.arange(x[0], x[-1]+1)
                interp_values = np.interp(interp_times, x, y)

                # Assign interpolated values to the corresponding positions
                interpolated_values[interp_times] = interp_values

            elif len(grp) == 1:
                # Only one valid data point, assign it to the correct position
                idx = data_times[grp[0]]
                interpolated_values[idx] = values[grp[0]]

        # Assign the interpolated values to the DataFrame
        interpolated_df.loc[company] = interpolated_values

    return interpolated_df

# Create an Excel writer using the default engine
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    for sheet_name in xls.sheet_names:
        df = xls.parse(sheet_name)
        interpolated_df = interpolate_sheet(df)
        if interpolated_df is not None:
            # Write to Excel without the index if you prefer
            interpolated_df.to_excel(writer, sheet_name=sheet_name)
            # Or include the company names as a column:
            # interpolated_df.reset_index().to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Interpolated data saved to: {output_file_path}")


Sheets in the Excel file: ['ROCE_Q', 'EBIT_Q', 'Total Activos_Q', 'Pasivos Corrientes_Q', 'Capital propio_Q', 'Deuda a LP_Q', 'ROA_Q', 'Beneficio neto_Q', 'ROC_Q', 'ROI_Q', 'EV_Q', 'Cap de mercado_Q', 'Deuda a CP_Q', 'Efectivo y equiv_Q', 'RORWA_Q']
Interpolated data saved to: C:\Users\Andy\OneDrive\Desktop\MCD\Tesis\Datos_fuente_Bloomberg\en valores\serie completa 2014-2024\Ratios\ratios_mensual_lineal.xlsx


### Transformación Cubic splining

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.interpolate import interp1d

# Corrected file paths (ensure they match your actual file locations)
file_path = Path('C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Ratios/en valores_ratios-trimestral_120 m.xlsx')
output_file_path = Path('C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Ratios/ratios_mensual_cubic_spline.xlsx')

# Verify that the input file exists
if not file_path.exists():
    raise FileNotFoundError(f"The file was not found at the specified path: {file_path}")

# Load the Excel file
xls = pd.ExcelFile(file_path)

print("Sheets in the Excel file:", xls.sheet_names)

def interpolate_sheet(sheet_df):
    periods = sheet_df.columns[1:]  # Ignore the first column (company names)
    companies = sheet_df.iloc[:, 0]  # Company names

    N = len(periods)

    # Generate labels starting with M3 (end of quarter)
    monthly_periods = []
    for t in range(N - 1):
        monthly_periods.append(f"{periods[t]}_M3")       # Data point at end of current quarter
        monthly_periods.append(f"{periods[t+1]}_M1")     # First month of next quarter
        monthly_periods.append(f"{periods[t+1]}_M2")     # Second month of next quarter
    # Append the last quarter's M3
    monthly_periods.append(f"{periods[N - 1]}_M3")       # Data point at end of last quarter

    total_months = len(monthly_periods)

    # Initialize DataFrame with companies as index and monthly_periods as columns
    interpolated_df = pd.DataFrame(index=companies, columns=monthly_periods)

    # Indices of data points (every third month)
    data_times = np.arange(0, total_months, 3)

    for i, company in enumerate(companies):
        # Replace non-numeric entries with np.nan and convert to numeric
        values = sheet_df.iloc[i, 1:].replace({'-': np.nan, '—': np.nan}).values
        values = pd.to_numeric(values, errors='coerce')

        # Initialize the interpolated values with NaNs
        interpolated_values = np.full(total_months, np.nan)

        # Get indices where data is valid
        valid_indices = np.where(~np.isnan(values))[0]

        # Loop over segments of consecutive valid data
        for grp in np.split(valid_indices, np.where(np.diff(valid_indices) != 1)[0]+1):
            if len(grp) > 1:
                # There are consecutive valid points to interpolate between
                start_idx = grp[0]
                end_idx = grp[-1]
                x = data_times[start_idx:end_idx+1]
                y = values[start_idx:end_idx+1]

                # Time indices for interpolation within this segment
                interp_times = np.arange(x[0], x[-1]+1)

                # Use cubic spline interpolation
                try:
                    cubic_interp = interp1d(x, y, kind='cubic')
                    interp_values = cubic_interp(interp_times)
                except Exception as e:
                    # Fallback to linear interpolation if cubic interpolation fails
                    interp_values = np.interp(interp_times, x, y)

                # Assign interpolated values to the corresponding positions
                interpolated_values[interp_times] = interp_values

            elif len(grp) == 1:
                # Only one valid data point, assign it to the correct position
                idx = data_times[grp[0]]
                interpolated_values[idx] = values[grp[0]]

        # Assign the interpolated values to the DataFrame
        interpolated_df.loc[company] = interpolated_values

    return interpolated_df

# Create an Excel writer using the default engine
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    for sheet_name in xls.sheet_names:
        df = xls.parse(sheet_name)
        interpolated_df = interpolate_sheet(df)
        if interpolated_df is not None:
            interpolated_df.to_excel(writer, sheet_name=sheet_name)

print(f"Interpolated data saved to: {output_file_path}")


Sheets in the Excel file: ['ROCE_Q', 'EBIT_Q', 'Total Activos_Q', 'Pasivos Corrientes_Q', 'Capital propio_Q', 'Deuda a LP_Q', 'ROA_Q', 'Beneficio neto_Q', 'ROC_Q', 'ROI_Q', 'EV_Q', 'Cap de mercado_Q', 'Deuda a CP_Q', 'Efectivo y equiv_Q', 'RORWA_Q']
Interpolated data saved to: C:\Users\Andy\OneDrive\Desktop\MCD\Tesis\Datos_fuente_Bloomberg\en valores\serie completa 2014-2024\Ratios\ratios_mensual_cubic_spline.xlsx
